# 📦 Notebook 01: Raw CSV Data Generation (Landing Stage)

## 🎯 Objectives
1. Generate synthetic, enterprise-scale e-commerce datasets containing **Orders**, **Customers**, and **Transactions**.
2. Save these datasets directly as **Raw CSV files** with headers into a storage landing directory (`/tmp/mini_project2/landing/` or Unity Catalog Volume path).
3. Introduce controlled dirty data (nulls, duplicate records, unformatted strings) to facilitate data quality testing in Silver transformations.

> **Note**: This notebook strictly creates raw CSV files. Ingestion into Delta format is handled in **Notebook 02**.

In [0]:
# Databricks notebook source
import random
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType, DateType

# Define base landing directory for Raw CSV files
LANDING_PATH = "/tmp/mini_project2/landing"
print(f"Target CSV Landing Path: {LANDING_PATH}")

In [0]:
# 1. Generate Synthetic Customers Raw Dataset (20,000 Records)
num_customers = 20000
regions = ["North_America", "EMEA", "APAC", "LATAM", "EUROPE"]
tiers = ["Standard", "Gold", "Platinum", "VIP"]

df_customers = spark.range(1, num_customers + 1).select(
    F.concat(F.lit("CUST_"), F.lpad(F.col("id"), 6, "0")).alias("customer_id"),
    F.concat(F.lit("Customer_"), F.col("id")).alias("customer_name"),
    F.concat(F.lit("cust_"), F.col("id"), F.lit("@example.com")).alias("email"),
    F.element_at(F.array(*[F.lit(r) for r in regions]), (F.rand() * len(regions) + 1).cast("int")).alias("region"),
    F.element_at(F.array(*[F.lit(t) for t in tiers]), (F.rand() * len(tiers) + 1).cast("int")).alias("tier"),
    F.date_add(F.to_date(F.lit("2023-01-01")), (F.rand() * 1000).cast("int")).alias("signup_date")
)

# Inject some nulls in region for data quality testing
df_customers = df_customers.withColumn(
    "region",
    F.when(F.rand() < 0.05, F.lit(None)).otherwise(F.col("region"))
)

# Write Customers to Raw CSV
customers_csv_path = f"{LANDING_PATH}/customers"
df_customers.coalesce(1).write.format("csv").option("header", "true").mode("overwrite").save(customers_csv_path)
print(f"✅ Successfully written raw Customers CSV to: {customers_csv_path}")

In [0]:
# 2. Generate Synthetic Orders Raw Dataset (100,000 Records)
num_orders = 100000
payment_methods = ["Credit_Card", "PayPal", "UPI", "Bank_Transfer", "Crypto"]
statuses = ["COMPLETED", "PENDING", "CANCELLED", "REFUNDED"]

df_orders = spark.range(1, num_orders + 1).select(
    F.concat(F.lit("ORD_"), F.lpad(F.col("id"), 7, "0")).alias("order_id"),
    F.concat(F.lit("CUST_"), F.lpad((F.rand() * num_customers + 1).cast("int"), 6, "0")).alias("customer_id"),
    F.date_add(F.to_date(F.lit("2024-01-01")), (F.rand() * 500).cast("int")).alias("order_date"),
    F.round(F.rand() * 500 + 10, 2).alias("total_amount"),
    F.element_at(F.array(*[F.lit(p) for p in payment_methods]), (F.rand() * len(payment_methods) + 1).cast("int")).alias("payment_method"),
    F.element_at(F.array(*[F.lit(s) for s in statuses]), (F.rand() * len(statuses) + 1).cast("int")).alias("status")
)

# Inject synthetic duplicates and missing order amounts for quarantine testing
df_orders_dirty = df_orders.withColumn(
    "total_amount",
    F.when(F.rand() < 0.03, F.lit(None)).otherwise(F.col("total_amount"))
)

# Append 2,000 duplicate order rows
df_duplicates = df_orders.limit(2000)
df_orders_final = df_orders_dirty.union(df_duplicates)

# Write Orders to Raw CSV
orders_csv_path = f"{LANDING_PATH}/orders"
df_orders_final.coalesce(1).write.format("csv").option("header", "true").mode("overwrite").save(orders_csv_path)
print(f"✅ Successfully written raw Orders CSV ({df_orders_final.count()} rows) to: {orders_csv_path}")

In [0]:
# 3. Generate Synthetic Transactions Raw Dataset (150,000 Records)
num_txns = 150000
txn_statuses = ["SUCCESS", "FAILED", "PROCESSING"]

df_txns = spark.range(1, num_txns + 1).select(
    F.concat(F.lit("TXN_"), F.lpad(F.col("id"), 8, "0")).alias("txn_id"),
    F.concat(F.lit("ORD_"), F.lpad((F.rand() * num_orders + 1).cast("int"), 7, "0")).alias("order_id"),
    F.round(F.rand() * 500 + 10, 2).alias("txn_amount"),
    F.from_unixtime(F.unix_timestamp(F.lit("2024-01-01 00:00:00")) + (F.rand() * 86400 * 500).cast("int")).alias("txn_timestamp"),
    F.element_at(F.array(*[F.lit(s) for s in txn_statuses]), (F.rand() * len(txn_statuses) + 1).cast("int")).alias("txn_status")
)

# Write Transactions to Raw CSV
txns_csv_path = f"{LANDING_PATH}/transactions"
df_txns.coalesce(1).write.format("csv").option("header", "true").mode("overwrite").save(txns_csv_path)
print(f"✅ Successfully written raw Transactions CSV to: {txns_csv_path}")

In [0]:
# 4. Summary of Generated Raw CSV Datasets
print("=== RAW LANDING DIRECTORY CONTENTS ===")
for path in [customers_csv_path, orders_csv_path, txns_csv_path]:
    file_list = dbutils.fs.ls(path)
    csv_files = [f.path for f in file_list if f.path.endswith(".csv")]
    print(f"Directory {path}: Found {len(csv_files)} CSV file(s) -> {csv_files[:1]}")